In [1]:
import re
import unicodedata
import pandas as pd

In [2]:
FANTASY_CSV = "fantasy_optimizer.csv"
STATS_CSV = "players_data-2025_2026.csv"
OUT_CSV = "fantasy_enriched.csv"

MAX_EDIT_DIST = 1

In [3]:
fantasy = pd.read_csv(FANTASY_CSV)
print(f"Total players: {len(fantasy)}")
print(fantasy["status"].value_counts().to_string())

fantasy = fantasy[fantasy["status"] != "transferred"].reset_index(drop=True)
print(f"\nAfter removing transferred: {len(fantasy)}")
print(fantasy.groupby("team")["name"].count().sort_values(ascending=False).to_string())

stats = pd.read_csv(STATS_CSV)

Total players: 134
status
playing        102
transferred     30
injured          1
suspended        1

After removing transferred: 104
team
Argentina    26
England      26
France       26
Spain        26


In [4]:
def norm(s):
    if pd.isna(s): # normalize text into lower case no accents
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z ]", "", re.sub(r"\s+", " ", s)).strip()

def lev(a, b):    # computing Levenshtein distance
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for ca in a:
        curr = [prev[0] + 1]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j-1] + (ca != cb)))
        prev = curr
    return prev[-1]

def best_match(name, pool):    # finds closest match in pool for name using Levenshtein distance
    fn = norm(name)
    best_dist, best_idx = MAX_EDIT_DIST + 1, None
    for idx, cn in zip(pool.index, pool["_name_norm"]):
        if abs(len(cn) - len(fn)) > MAX_EDIT_DIST:
            continue
        d = lev(fn, cn)
        if d < best_dist:
            best_dist, best_idx = d, idx
    if best_idx is not None and best_dist <= MAX_EDIT_DIST:
        return best_idx, best_dist
    return None, None


In [5]:

stats["fbref_code"] = stats["Nation"].str.extract(r"([A-Z]{2,4})$") #  new column fbref_code

COUNT_COLS = [c for c in [
    "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "xG", "xAG", "npxG", "G-PK",
    "Tkl", "TklW", "Blocks", "Int", "Tkl+Int", "Clr", "Err",
    "PrgP", "PrgC", "KP", "PPA", "xA", "Ast_stats_passing",
    "GA", "Saves", "CS", "PKA", "PKsv",
    "Touches", "Carries", "PrgR", "Mis", "Dis",
    "CrdY", "CrdR", "PKwon", "PKcon", "Recov",
    "PK", "PKatt", "SoTA", "PKm", "Sh", "SoT",
    "Fls", "Fld", "Off", "Crs", "OG", "2CrdY",
] if c in stats.columns]

RATE_COLS = [c for c in [
    "Cmp%_stats_passing", "Save%", "CS%",
    "GA90", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT",
] if c in stats.columns]

META_COLS = [c for c in ["Pos", "Squad", "Comp", "Age"] if c in stats.columns]

agg_dict = {c: "sum" for c in COUNT_COLS} # sums integer columns
agg_dict.update({c: "first" for c in META_COLS}) # takes first value for meta columns

if "Squad" in META_COLS:
    agg_dict["Squad"] = lambda x: " / ".join(x.dropna().astype(str).unique())


stats_agg = stats.groupby(["Player", "fbref_code"], as_index=False, sort=False).agg(agg_dict)

rates = stats.loc[
    stats.groupby(["Player", "fbref_code"])["MP"].idxmax(),
    ["Player", "fbref_code"] + RATE_COLS   # preserves rates of club where player played most
]

stats_agg = stats_agg.merge(rates, on=["Player", "fbref_code"], how="left")

stats_agg["fbref_multi_club"] = stats.groupby(["Player", "fbref_code"]).size().gt(1).values # players with multiple clubs in the same season

stats_agg["_name_norm"] = stats_agg["Player"].apply(norm) #


print(f"Stats aggregated: {len(stats_agg)} players")

Stats aggregated: 2685 players


In [6]:
NATION_MAP = {
    "Algeria": "ALG", "Argentina": "ARG", "Australia": "AUS", "Austria": "AUT",
    "Belgium": "BEL", "Bosnia and Herzegovina": "BIH", "Brazil": "BRA",
    "Cabo Verde": "CPV", "Canada": "CAN", "Colombia": "COL", "Congo DR": "COD",
    "Croatia": "CRO", "Curacao": "CUW", "Czechia": "CZE", "Ecuador": "ECU",
    "Egypt": "EGY", "England": "ENG", "France": "FRA", "Germany": "GER",
    "Ghana": "GHA", "Haiti": "HAI", "IR Iran": "IRN", "Iraq": "IRQ",
    "Japan": "JPN", "Jordan": "JOR", "Korea Republic": "KOR", "Mexico": "MEX",
    "Morocco": "MAR", "Netherlands": "NED", "New Zealand": "NZL", "Norway": "NOR",
    "Panama": "PAN", "Paraguay": "PAR", "Portugal": "POR", "Qatar": "QAT",
    "Saudi Arabia": "KSA", "Scotland": "SCO", "Senegal": "SEN",
    "South Africa": "RSA", "Spain": "ESP", "Sweden": "SWE", "Switzerland": "SUI",
    "Tunisia": "TUN", "Turkiye": "TUR", "Uruguay": "URU", "USA": "USA",
    "Uzbekistan": "UZB", "Cote d Ivoire": "CIV",
}

fantasy["fbref_code"] = fantasy["team"].map(NATION_MAP)

unmapped = fantasy.loc[fantasy["fbref_code"].isna(), "team"].dropna().unique()
if len(unmapped):
    print(f"\nUnmapped teams: {list(unmapped)}")
else:
    print("\n No team unmapped")


 No team unmapped


/tmp/ipykernel_31348/944440591.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fantasy["fbref_code"] = fantasy["team"].map(NATION_MAP)


In [7]:
codes = set(fantasy["fbref_code"].dropna().unique())

stats_filtered = stats_agg[stats_agg["fbref_code"].isin(codes)].copy()

print(f"Stats rows for tournament nations: {len(stats_filtered)}")   # keep only players stats whose nation in the world cup.

Stats rows for tournament nations: 975


In [8]:
ADD_COLS = COUNT_COLS + RATE_COLS + META_COLS + ["fbref_multi_club"]
for col in ADD_COLS:
    fantasy[f"club_{col}"] = pd.NA
fantasy["clubstats_matched_player"] = pd.NA
fantasy["clubs_match_dist"] = pd.NA

match_log = []  # match log

for code, grp in fantasy.groupby("fbref_code", dropna=True):
    pool = stats_filtered[stats_filtered["fbref_code"] == code]  # only matches inside nation
    if pool.empty:
        continue
    for idx in grp.index:
        fname = fantasy.at[idx, "name"]
        match_idx, dist = best_match(fname, pool)
        if match_idx is None:
            continue
        row = stats_filtered.loc[match_idx]
        for col in ADD_COLS:
            if col in row.index:
                fantasy.at[idx, f"club_{col}"] = row[col]
        fantasy.at[idx, "clubstats_matched_player"] = row["Player"]
        fantasy.at[idx, "clubs_match_dist"] = dist
        match_log.append({"nation": code, "fantasy": fname, "club": row["Player"], "dist": dist})

/tmp/ipykernel_31348/2046871141.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fantasy[f"club_{col}"] = pd.NA
/tmp/ipykernel_31348/2046871141.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fantasy[f"club_{col}"] = pd.NA
/tmp/ipykernel_31348/2046871141.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

In [9]:
matched = fantasy["clubs_match_dist"].notna().sum()
print(f"\nMatched {matched} out of {len(fantasy)} ({matched/len(fantasy):.1%})")

match_df = pd.DataFrame(match_log)
if not match_df.empty:
    for d in range(MAX_EDIT_DIST + 1):
        print(f"  dist={d}: {(match_df['dist']==d).sum()}")

    suspect = match_df[match_df["dist"] > 0].sort_values(["dist","nation"])
    if not suspect.empty:
        print(f"\nImperfect matches (dist > 0)")
        print(suspect[["nation","fantasy","club","dist"]].to_string(index=False))

print(f"\nMatch rate by nation:")
for code, grp in fantasy.groupby("fbref_code", dropna=True):
    n_matched = grp["clubs_match_dist"].notna().sum()
    n_total = len(grp)
    flag = " no top-5 data" if n_matched == 0 else ""
    print(f"  {code}: {n_matched} in {n_total} ({n_matched/n_total:.0%}){flag}")

unmatched = fantasy[fantasy["clubs_match_dist"].isna()][["name","team","position"]]


Matched 91 out of 104 (87.5%)
  dist=0: 90
  dist=1: 1

Imperfect matches (dist > 0)
nation     fantasy        club  dist
   ESP Yéremy Pino Yeremi Pino     1

Match rate by nation:
  ARG: 18 in 26 (69%)
  ENG: 25 in 26 (96%)
  ESP: 24 in 26 (92%)
  FRA: 24 in 26 (92%)


In [10]:
fantasy = fantasy.drop(columns=["fbref_code"], errors="ignore")

fantasy.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {fantasy.shape[0]} rows x {fantasy.shape[1]} cols")



Saved fantasy_enriched.csv 104 rows x 214 cols
